In [0]:
print("Hello World")

- **freight-price**: Price charged by the transporter to transport the product
- **product_weight_g**: Product weight in grams
- **product_score**: Average Product rating
- **customers**: No.of customers in that category
- **weekdays**: No.of weekdays in that month
- **weekends**: No.of weekends in that month
- **holidays**: No.of holidays in that month
- **lag_price**: Previous month price of the product

- Change column names of `product_name_lenght`, `product_description_lenght`
- Better FeatureEng for **weekday, weekend, holiday**

In [0]:
ft_path = "mlcore_dev.retail_prices_1303_v1.transformed_retail_prices_1303_v1_ft"
ft_df = spark.sql(f"SELECT * FROM {ft_path} LIMIT 676")
ft_df.display()

In [0]:
df = ft_df.toPandas()
df

In [0]:
# A New Column: Avg unit_price, which will contain the mean of unit_price for each group of same month, product_category_name.
df["avg_unit_price_by_product"] = df.groupby(["month", "product_category_name"])["unit_price"].transform("mean")

# A New Column: Avg customers, which will contain the mean number of customers for each group of same month, product_category_name.
df["avg_customers_by_product"] = df.groupby(["month", "product_category_name"])["customers"].transform("mean")

# A New Column: Average Competitor Prices, which will contain the mean of 3 competitor prices for each group of same month, product_category_name.
df["avg_comp_price_by_product"] = (
    df[["comp_1", "comp_2", "comp_3"]].mean(axis=1)
)

df["avg_comp_price_by_product"] = df.groupby(["month", "product_category_name"])["avg_comp_price_by_product"].transform("mean")

round_cols = ["freight_price", "unit_price", "s", "comp_1", "comp_2", "comp_3", "lag_price", "avg_comp_price_by_product", "avg_customers_by_product", "avg_unit_price_by_product"]

# Roudning off float value columns to 2 digits.
for col in round_cols:
    df[col] = df[col].round(2)

df

In [0]:
ft_df["lag_units_sold"] = ft_df.groupby("product_category_name")["units_sold"].shift(1)
ft_df

In [0]:
ft_df.count()

In [0]:
gt_path = "mlcore_dev.retail_prices_1303_v1.transformed_retail_prices_1303_v1_gt"
gt_df = spark.sql(f"SELECT * FROM {gt_path} LIMIT 676")
gt_df.display()

In [0]:
ft_df = ft_df.toPandas()
# gt_df = gt_df.toPandas()
ft_df.columns

In [0]:
# Change back to normal year by adding 2017
ft_df["year"] = ft_df["year"] + 2017
# ft_df

In [0]:
# # Add time index
ft_df["time_idx"] = ft_df["year"] * 12 + ft_df["month"]
ft_df["time_idx"] -= ft_df["time_idx"].min()
ft_df.display()

In [0]:
ft_df["time_idx"].unique()

In [0]:
ft_df.columns

1. Groupby `product_category_name`, `time_idx` -> Avg unit_price, Avg of 3 competitors prices, Avg customers

In [0]:
df = spark.sql(f"SELECT * FROM mlcore_dev.retail_prices_1503_v2.train_output_retail_prices_1503_v2")
df.display()
df = df.toPandas()

In [0]:
df[(df["freight_price"] == 14.84) & (df["product_category_name"] == "bed_bath_table")].display()

In [0]:
%pip install xgboost mlflow
!pip install --upgrade mlflow typing_extensions

In [0]:
import xgboost as xgb
from sklearn.metrics import *
import pandas as pd
from sklearn.model_selection import KFold
xgb.__version__
# import mlflow

In [0]:
df = df.iloc[:, 0:20]
df.display()

In [0]:
class DemandForecastingModel():
    """
    DemandForecasting
    """
    def __init__(self):
        self.model = None
        self.pcn_encode_dict = {
            'bed_bath_table': 0,
            'garden_tools': 1,
            'consoles_games': 2,
            'health_beauty': 3,
            'cool_stuff': 4,
            'perfumery': 5,
            'computers_accessories': 6,
            'watches_gifts': 7,
            'furniture_decor': 8
        }
        self.round_cols = ["freight_price", "unit_price", "s", "comp_1", "comp_2", "comp_3", "lag_price", "avg_comp_price_by_product", "avg_customers_by_product", "avg_unit_price_by_product"]
    
    def feature_engineering(self, df):
        """Applying Feature Engineering to the input DataFrame."""
        
        df = df.copy()  # Avoid modifying original DataFrame
        # Encoding categorical column
        df["product_category_name"] = df["product_category_name"].map(self.pcn_encode_dict)

        # A New Column: Avg unit_price, which will contain the mean of unit_price for each group of same month, product_category_name.
        df["avg_unit_price_by_product"] = df.groupby(["month", "product_category_name"])["unit_price"].transform("mean")

        # A New Column: Avg customers, which will contain the mean number of customers for each group of same month, product_category_name.
        df["avg_customers_by_product"] = df.groupby(["month", "product_category_name"])["customers"].transform("mean")

        # A New Column: Average Competitor Prices, which will contain the mean of 3 competitor prices for each group of same month, product_category_name.
        df["avg_comp_price_by_product"] = df.groupby(["month", "product_category_name"])[["comp_1", "comp_2", "comp_3"]].transform("mean").mean(axis=1)

        # Roudning off float value columns to 2 digits.
        for col in self.round_cols:
            df[col] = df[col].round(2)

        return df
    
    def train(self, X_train, y_train, n_splits=5):
        """Performs feature engineering and trains the XGBoost model."""
        X_train = self.feature_engineering(X_train)
        # Train XGBoost Model
        self.model = xgb.XGBRegressor(
            objective='reg:squarederror',
            max_depth=5,
            learning_rate=0.3,
            n_estimators=500,
            reg_lambda=10,            # L2 regularization
            reg_alpha=2,              # L1 regularization
            eval_metric='rmse'
        )
        self.model.fit(X_train, y_train)
    
    def predict(self, X_test):
        """Applies the trained model on new data."""
        X_test = self.feature_engineering(X_test)
        # Ensure model is trained
        if self.model is None:
            raise ValueError("Model has not been trained yet. Call `train()` first.")

        return self.model.predict(X_test)
        

In [0]:
# Split the Data to Train and Test
test_size = 0.2
traindf = df.iloc[int(df.shape[0] * test_size):]
testdf = df.iloc[:int(df.shape[0] * test_size)]

target_columns = ["units_sold"]

X_train = traindf.drop(columns=target_columns)
y_train = traindf[target_columns[0]]

X_test = testdf.drop(columns=target_columns)
y_test = testdf[target_columns[0]]

# Create model instance.
model = DemandForecastingModel()

# Train the model
model.train(X_train, y_train)

y_pred_train = model.predict(X_test=X_train)
y_pred = model.predict(X_test=X_test)

In [0]:
# Predict it on Test and calculate metrics
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)

test_metrics = {"r2":r2, "mse":mse, "mae":mae, "rmse":rmse}

r2 = r2_score(y_train, y_pred_train)
mse = mean_squared_error(y_train, y_pred_train)
mae = mean_absolute_error(y_train, y_pred_train)
rmse = mean_squared_error(y_train, y_pred_train, squared=False)

train_metrics = {"r2":r2, "mse":mse, "mae":mae, "rmse":rmse}

print(train_metrics)
print(test_metrics)

In [0]:
pred_train = traindf
pred_train["prediction"] = y_pred_train
pred_train["dataset_type_71E4E76EB8C12230B6F51EA2214BD5FE"] = "train"

pred_test = testdf
pred_test["prediction"] = y_pred
pred_test["dataset_type_71E4E76EB8C12230B6F51EA2214BD5FE"] = "test"

final_train_output_df = pd.concat([pred_train, pred_test])
train_output_df = spark.createDataFrame(final_train_output_df)
train_output_df.display